# Do foundation models beat classical pipelines on small scRNA-seq datasets?

A reproducible benchmark of cell embeddings for **batch integration** and **cell-type annotation** on the scIB human pancreas data (7 batches, 14,252 cells, 14 cell types), with a PBMC3k sanity check.

| Family | Method |
|---|---|
| Classical | Scanpy PCA, PCA + Harmony |
| Deep generative | scVI (3 training seeds) |
| Foundation model | Geneformer V2-104M, zero-shot (all genes, plus a 2,000-HVG sensitivity run) |

**How to run on Kaggle:** Settings → Internet **ON**, Accelerator **GPU (T4)** → Run All. The full run takes roughly 2 to 3 hours (estimated from the timings of the individual steps). Every arm saves its results to `results/` (CSV, small `.npy`) or `/kaggle/temp/cache` (large embeddings) and **reloads them if they exist**, so a restart does not recompute finished work.

**Structure:** setup and helper module → PBMC3k sanity check → pancreas data audit → classical arm → scVI arm → Geneformer arm and sensitivity checks → results (tables, figures) → biological interpretation → conclusions.

All helper code lives in `scbench.py` (written by the next cell, copy it to `src/` in the repository).

## 0. Setup

In [ ]:
# Pinned ranges as used for the reported results. Do NOT reinstall numpy / pandas / scikit-learn / torch on Kaggle.
!pip install -q "scanpy>=1.10,<1.12" "anndata>=0.10" harmonypy leidenalg python-igraph scikit-misc scvi-tools==1.5.2

In [ ]:
%%writefile /kaggle/working/scbench.py
"""scbench: helper functions for the scRNA-seq embedding benchmark.

Do foundation models beat classical pipelines on small scRNA-seq datasets?
Compares PCA, PCA + Harmony, scVI and zero-shot Geneformer embeddings with scIB-style metrics.

Everything the notebook needs is here (data loading, preprocessing, embeddings helpers,
metrics, plotting), so the notebook itself stays a readable narrative.
"""
import os, random, gc, hashlib

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score, silhouette_score,
                             silhouette_samples, f1_score, precision_recall_fscore_support)
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------------------
SEED = 0
SEEDS = (0, 1, 2)                     # replicate seeds: Leiden, CV folds, and scVI training
CFG = dict(
    n_hvg=2000, n_pcs=50, n_neighbors=15, knn_k=15, cv_folds=5,
    resolutions=[0.05, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0, 1.4, 1.8],   # pancreas Leiden grid
    pbmc_resolutions=[0.2, 0.4, 0.6, 0.8, 1.0, 1.4, 1.8],              # grid used for the PBMC3k sanity check
    out_dir="/kaggle/working/results",
)


def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
    except ImportError:
        pass


# --------------------------------------------------------------------------------------
# Data loading
# --------------------------------------------------------------------------------------
def load_pbmc3k():
    """Raw counts from pbmc3k + cell-type labels transferred from pbmc3k_processed (by barcode)."""
    raw = sc.datasets.pbmc3k()
    lab = sc.datasets.pbmc3k_processed()
    keep = raw.obs_names.intersection(lab.obs_names)
    raw = raw[keep].copy()
    raw.obs["cell_type"] = lab.obs.loc[keep, "louvain"].astype(str).values
    raw.var_names_make_unique()
    raw.layers["counts"] = raw.X.copy()
    raw.obs["batch"] = "pbmc3k"
    return raw


def figshare_download(article_id, fname, dest):
    """Download one file from a figshare article through the public API and verify its MD5."""
    import requests
    art = requests.get(f"https://api.figshare.com/v2/articles/{article_id}", timeout=60).json()
    print("article:", art["title"], "| version:", art.get("version"), "| doi:", art.get("doi"))
    f = {x["name"]: x for x in art["files"]}[fname]
    if not os.path.exists(dest):
        with requests.get(f["download_url"], stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(dest, "wb") as out:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    out.write(chunk)
    md5 = hashlib.md5(open(dest, "rb").read()).hexdigest()
    print("md5 match:", md5 == f["computed_md5"], "|", md5)
    assert md5 == f["computed_md5"], "Checksum mismatch: delete the file and re-run."


def load_pancreas(path, batch_col, label_col, drop_batches, round_batches, max_cells=None, seed=SEED):
    """scIB pancreas: drop batches whose `counts` layer is not count-like, round near-integer batches,
    verify every remaining value is a non-negative integer, return a clean sparse-count AnnData."""
    a = sc.read_h5ad(path)
    b_all = a.obs[batch_col].astype(str)
    y_all = a.obs[label_col].astype(str)
    keep = ~b_all.isin(drop_batches).values
    counts = np.asarray(a.layers["counts"])[keep]
    obs = pd.DataFrame({"batch": b_all[keep].values, "cell_type": y_all[keep].values}, index=a.obs_names[keep])
    var_names = a.var_names.astype(str)
    lost = sorted(set(y_all) - set(obs["cell_type"]))
    n_removed = int((~keep).sum())
    del a
    gc.collect()
    print(f"dropped batches {drop_batches}: {n_removed} cells removed -> {obs.shape[0]} remain")
    print("cell types lost entirely by the exclusion:", lost or "none")

    rb = obs["batch"].isin(round_batches).values
    if rb.any():
        before = counts[rb]
        after = np.round(before)
        print(f"rounded {int(rb.sum())} cells in {round_batches}; max |change| = {np.abs(after - before).max():.3f}")
        counts[rb] = after
        del before, after

    assert counts.min() >= 0, "negative values in counts"
    for i in range(0, counts.shape[0], 2000):
        ch = counts[i:i + 2000]
        bad = ~np.isclose(ch, np.round(ch))
        if bad.any():
            raise AssertionError("non-integer values remain in batches: "
                                 f"{obs['batch'].iloc[i:i + 2000][bad.any(1)].unique().tolist()}")

    X = sp.csr_matrix(counts)
    del counts
    gc.collect()
    out = ad.AnnData(X=X, obs=obs, var=pd.DataFrame(index=var_names))
    out.var_names_make_unique()
    out.layers["counts"] = out.X.copy()

    if max_cells and out.n_obs > max_cells:
        frac, idx = max_cells / out.n_obs, []
        for _, d in out.obs.groupby(["batch", "cell_type"], observed=True):
            idx += list(d.sample(n=max(1, int(round(len(d) * frac))), random_state=seed).index)
        out = out[idx].copy()
        print(f"subsampled to {out.n_obs} cells")
    return out


# --------------------------------------------------------------------------------------
# Preprocessing and classical embeddings
# --------------------------------------------------------------------------------------
def preprocess(adata, n_hvg=CFG["n_hvg"], n_pcs=CFG["n_pcs"], seed=SEED, hvg_batch_key=None, max_pct_mt=5):
    """QC -> HVG (seurat_v3 on raw counts, optionally batch-aware) -> normalize/log1p -> scale -> PCA.
    Returns the HVG-subset AnnData; the full-gene log-normalised matrix is kept in `.raw`."""
    adata = adata.copy()
    adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
    n0 = adata.n_obs
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    if max_pct_mt is not None:
        adata = adata[adata.obs["pct_counts_mt"] < max_pct_mt].copy()
    print(f"cells: {n0} -> {adata.n_obs} | genes: {adata.n_vars}")
    sc.pp.highly_variable_genes(adata, n_top_genes=n_hvg, flavor="seurat_v3",
                                layer="counts", batch_key=hvg_batch_key)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.raw = adata
    adata = adata[:, adata.var["highly_variable"]].copy()
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=n_pcs, svd_solver="arpack", random_state=seed)
    return adata


def run_harmony(adata, batch_key, basis="X_pca", out_key="X_pca_harmony", seed=SEED):
    import harmonypy
    try:
        ho = harmonypy.run_harmony(adata.obsm[basis], adata.obs, batch_key, random_state=seed)
    except TypeError:
        ho = harmonypy.run_harmony(adata.obsm[basis], adata.obs, batch_key)
    Z = np.asarray(ho.Z_corr)
    if Z.shape[0] != adata.n_obs:                     # harmonypy has returned both orientations across versions
        Z = Z.T
    assert Z.shape == adata.obsm[basis].shape, Z.shape
    adata.obsm[out_key] = Z


def graph_only(adata, rep, name, n_neighbors=CFG["n_neighbors"], seed=SEED):
    sc.pp.neighbors(adata, use_rep=rep, n_neighbors=n_neighbors, key_added=name, random_state=seed)


def graph_umap(adata, rep, name, n_neighbors=CFG["n_neighbors"], seed=SEED):
    graph_only(adata, rep, name, n_neighbors, seed)
    sc.tl.umap(adata, neighbors_key=name, random_state=seed)
    adata.obsm[f"X_umap_{name}"] = adata.obsm["X_umap"].copy()


def ensure_graph(adata, rep, name, umap=False):
    """Build the kNN graph (and optionally the UMAP) only if it does not exist yet."""
    if f"{name}_connectivities" not in adata.obsp:
        (graph_umap if umap else graph_only)(adata, rep, name)
    elif umap and f"X_umap_{name}" not in adata.obsm:
        graph_umap(adata, rep, name)


# --------------------------------------------------------------------------------------
# Metrics (scIB-style)
# --------------------------------------------------------------------------------------
def batch_asw(X, batches, labels):
    """scIB batch ASW: per cell type, 1 - |silhouette of batch labels|, averaged. 1 = well mixed."""
    scores = []
    for lab in np.unique(labels):
        m = labels == lab
        nb = len(np.unique(batches[m]))
        if nb < 2 or m.sum() <= nb:
            continue                                   # cell type lives in one batch: not scoreable
        scores.append((1 - np.abs(silhouette_samples(X[m], batches[m]))).mean())
    return float(np.mean(scores))


def graph_connectivity(adata, name, label_key="cell_type"):
    """scIB graph connectivity: per cell type, fraction in the largest connected component of the kNN graph."""
    A, y, scores = adata.obsp[f"{name}_connectivities"], adata.obs[label_key].values, []
    for lab in np.unique(y):
        idx = np.where(y == lab)[0]
        _, comp = connected_components(A[idx][:, idx], directed=False)
        scores.append(np.bincount(comp).max() / len(idx))
    return float(np.mean(scores))


def cluster_scores(adata, name, label_key="cell_type", seed=SEED, resolutions=None):
    """Leiden resolution sweep (best NMI against the labels, scIB convention) for one seed."""
    resolutions = resolutions or CFG["resolutions"]
    best = None
    for r in resolutions:
        sc.tl.leiden(adata, resolution=r, neighbors_key=name, key_added="_tmp_leiden",
                     flavor="igraph", n_iterations=2, directed=False, random_state=seed)
        cl, y = adata.obs["_tmp_leiden"], adata.obs[label_key]
        nmi = normalized_mutual_info_score(y, cl)
        if best is None or nmi > best["NMI"]:
            best = {"NMI": nmi, "ARI": adjusted_rand_score(y, cl), "best_res": r, "n_clusters": int(cl.nunique())}
    del adata.obs["_tmp_leiden"]
    return best


def knn_macro_f1(X, y, k=CFG["knn_k"], folds=CFG["cv_folds"], seed=SEED):
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    pred = cross_val_predict(KNeighborsClassifier(n_neighbors=k), X, y, cv=cv)
    return f1_score(y, pred, average="macro")


def _lr():
    """Class-balanced logistic regression on standardised features (so PCA, scVI and Geneformer
    embeddings of different scale are treated alike)."""
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))


def annot_cv(X, y, seed, folds=CFG["cv_folds"]):
    """Annotation with stratified CV: kNN and balanced logistic regression macro-F1."""
    cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=seed)
    pred = cross_val_predict(_lr(), X, y, cv=cv, n_jobs=-1)
    return {"kNN_macroF1": knn_macro_f1(X, y, seed=seed), "LR_macroF1": f1_score(y, pred, average="macro")}


def annot_lobo(X, y, b, labels=None):
    """Leave-one-batch-out: train on all other batches, predict the held-out one (pooled macro-F1)."""
    pred = cross_val_predict(_lr(), X, y, cv=LeaveOneGroupOut(), groups=b, n_jobs=-1)
    return f1_score(y, pred, labels=labels, average="macro")


def _geometry_metrics(adata, rep, name, label_key, batch_key):
    X, y, b = adata.obsm[rep], adata.obs[label_key].values, adata.obs[batch_key].values
    return X, y, b, {"cell_type_ASW": (silhouette_score(X, y) + 1) / 2,
                     "batch_ASW": batch_asw(X, b, y),
                     "graph_conn": graph_connectivity(adata, name, label_key)}


def evaluate_all(adata, rep, name, label_key="cell_type", batch_key="batch", seeds=SEEDS):
    """Full evaluation, one row per seed. The seed varies Leiden and the CV folds;
    ASW, graph connectivity and leave-one-batch-out are deterministic given the embedding."""
    X, y, b, fixed = _geometry_metrics(adata, rep, name, label_key, batch_key)
    fixed["LR_LOBO_macroF1"] = annot_lobo(X, y, b)
    rows = [{"method": name, "seed": s, **cluster_scores(adata, name, label_key, s),
             **annot_cv(X, y, s), **fixed} for s in seeds]
    return pd.DataFrame(rows)


def evaluate_light(adata, rep, name, label_key="cell_type", batch_key="batch", seeds=SEEDS):
    """Clustering + geometry + kNN only (no logistic regression); used for sensitivity runs."""
    X, y, b, fixed = _geometry_metrics(adata, rep, name, label_key, batch_key)
    return pd.DataFrame([{"method": name, "seed": s, **cluster_scores(adata, name, label_key, s),
                          "kNN_macroF1": knn_macro_f1(X, y, seed=s), **fixed} for s in seeds])


def evaluate_bio(adata, rep, name, label_key="cell_type", seed=SEED, resolutions=None):
    """Single-batch evaluation (PBMC3k): ARI, NMI, cell-type ASW, kNN macro-F1."""
    X, y = adata.obsm[rep], adata.obs[label_key].values
    return {"method": name, **cluster_scores(adata, name, label_key, seed, resolutions),
            "cell_type_ASW": (silhouette_score(X, y) + 1) / 2, "kNN_macroF1": knn_macro_f1(X, y, seed=seed)}


def summarize(per_seed):
    """Mean, sd and 'mean ± sd' tables over seeds, per method."""
    cols = [c for c in per_seed.columns if c not in ("method", "seed", "best_res")]
    g = per_seed.groupby("method")[cols]
    mean, sd = g.mean(), g.std(ddof=1)
    return mean.round(3), sd.round(3), mean.round(3).astype(str) + " ± " + sd.round(3).astype(str)


def per_class_table(adata, reps, label_key="cell_type", seed=SEED):
    """Per-cell-type F1 (kNN and balanced LR, stratified CV) for each embedding; also returns the kNN predictions."""
    y = adata.obs[label_key].values
    labs = np.unique(y)
    cv = StratifiedKFold(n_splits=CFG["cv_folds"], shuffle=True, random_state=seed)
    rows, preds = {}, {}
    for m, rep in reps.items():
        X = adata.obsm[rep]
        preds[m] = cross_val_predict(KNeighborsClassifier(n_neighbors=CFG["knn_k"]), X, y, cv=cv)
        pl = cross_val_predict(_lr(), X, y, cv=cv, n_jobs=-1)
        rows[f"{m}_kNN"] = precision_recall_fscore_support(y, preds[m], labels=labs, zero_division=0)[2]
        rows[f"{m}_LR"] = precision_recall_fscore_support(y, pl, labels=labs, zero_division=0)[2]
    table = pd.DataFrame(rows, index=labs).round(2)
    table["n_cells"] = pd.Series(y).value_counts()
    return table, preds


def per_type_connectivity_mixing(adata, reps, graph_names, label_key="cell_type", batch_key="batch"):
    """Per cell type: graph connectivity and batch mixing for each embedding (what was preserved or lost)."""
    y, b = adata.obs[label_key].values, adata.obs[batch_key].values
    cts = sorted(set(y))
    conn, mix = {}, {}
    for m, rep in reps.items():
        A, X = adata.obsp[f"{graph_names[m]}_connectivities"], adata.obsm[rep]
        conn[m], mix[m] = {}, {}
        for ct in cts:
            idx = np.where(y == ct)[0]
            _, comp = connected_components(A[idx][:, idx], directed=False)
            conn[m][ct] = np.bincount(comp).max() / len(idx)
            nb = len(np.unique(b[idx]))
            mix[m][ct] = (1 - np.abs(silhouette_samples(X[idx], b[idx])).mean()) if nb >= 2 and len(idx) > nb else np.nan
    n_ct = pd.Series(y).value_counts()
    return (pd.DataFrame(conn).round(2).assign(n_cells=n_ct).sort_values("n_cells"),
            pd.DataFrame(mix).round(2).assign(n_cells=n_ct).sort_values("n_cells"))


# --------------------------------------------------------------------------------------
# Plotting
# --------------------------------------------------------------------------------------
def plot_umaps(adata, names, colors=("cell_type", "batch"), save=None):
    fig, axes = plt.subplots(len(colors), len(names), figsize=(4.5 * len(names), 4 * len(colors)), squeeze=False)
    for j, n in enumerate(names):
        for i, c in enumerate(colors):
            key = f"leiden_{n}" if c == "leiden" else c
            sc.pl.embedding(adata, basis=f"umap_{n}", color=key, ax=axes[i, j], show=False,
                            title=f"{n} | {c}", legend_loc="right margin" if j == len(names) - 1 else None)
    plt.tight_layout()
    if save:
        fig.savefig(f"{CFG['out_dir']}/{save}", dpi=150, bbox_inches="tight")
    plt.show()


def metric_bar(per_seed, metrics, order, save=None):
    g = per_seed.groupby("method")[metrics]
    mean, sd = g.mean(), g.std(ddof=1).fillna(0)
    x, w = np.arange(len(metrics)), 0.8 / len(order)
    fig, ax = plt.subplots(figsize=(1.9 * len(metrics) + 2, 4.5))
    for i, m in enumerate(order):
        ax.bar(x - 0.4 + w * (i + 0.5), mean.loc[m], w, yerr=sd.loc[m], capsize=2, label=m)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, rotation=30, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("score (higher is better)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    if save:
        fig.savefig(f"{CFG['out_dir']}/{save}", dpi=150, bbox_inches="tight")
    plt.show()


# --------------------------------------------------------------------------------------
# Geneformer zero-shot embeddings (own implementation of the rank-value encoding; the official
# package does not import under transformers 5.x)
# --------------------------------------------------------------------------------------
def build_gene_mapping(var_names, name2id, ens_map, tokdict, mediandict):
    """Gene symbol -> Ensembl ID (direct dictionary, then alias dictionary), restricted to the model
    vocabulary. Duplicate IDs are collapsed by summing counts (through the sparse matrix M: genes x vocab genes)."""
    genes = pd.Series(list(var_names), name="symbol")
    ens_direct = genes.map(name2id)
    ens = ens_direct.fillna(genes.str.upper().map(ens_map))
    in_vocab = ens.isin(set(tokdict))
    vocab_ens = sorted(set(ens[in_vocab]))
    index = {e: k for k, e in enumerate(vocab_ens)}
    rows = np.where(in_vocab.values)[0]
    M = sp.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, [index[e] for e in ens[in_vocab]])),
                      shape=(len(genes), len(vocab_ens)))
    tok_ids = np.array([tokdict[e] for e in vocab_ens])
    medians = np.array([mediandict.get(e, np.nan) for e in vocab_ens], dtype=np.float32)
    valid = np.isfinite(medians) & (medians > 0)
    return dict(genes=genes, ens=ens, ens_direct=ens_direct, in_vocab=in_vocab, M=M, index=index,
                tok_ids=tok_ids, medians=medians, valid=valid)


def rank_tokens(Cv, n_counts, tok_ids, medians, valid, cls_id, eos_id, max_len, target_sum=10_000):
    """Rank-value encoding: counts / n_counts * 1e4 / gene median, genes ranked in descending order,
    <cls> ... <eos>, truncated to max_len (cls and eos included)."""
    Cv, out = Cv.tocsr(), []
    for i in range(Cv.shape[0]):
        s, e = Cv.indptr[i], Cv.indptr[i + 1]
        idx, val = Cv.indices[s:e], Cv.data[s:e]
        keep = (val > 0) & valid[idx]
        idx, val = idx[keep], val[keep]
        order = np.argsort(-(val / n_counts[i] * target_sum / medians[idx]))
        out.append(np.concatenate([[cls_id], tok_ids[idx[order]][: max_len - 2], [eos_id]]).astype(np.int64))
    return out


def embed_tokens(model, tokens, pad_id, eos_id, bs=8, layer=-2, device="cuda"):
    """Cell embedding = mean over gene tokens (no pad/cls/eos) of the second-to-last layer
    (hidden_states[-1] is the last layer). Cells are processed in length order to limit padding."""
    import torch
    n, lens_ = len(tokens), np.array([len(t) for t in tokens])
    order = np.argsort(lens_)
    out = np.zeros((n, model.config.hidden_size), dtype=np.float32)
    with torch.no_grad():
        for k in range(0, n, bs):
            idx = order[k:k + bs]
            L = lens_[idx].max()
            ids = torch.full((len(idx), L), pad_id, dtype=torch.long)
            for r, i in enumerate(idx):
                ids[r, :lens_[i]] = torch.from_numpy(tokens[i])
            ids = ids.to(device)
            att = (ids != pad_id).long()
            with torch.autocast("cuda", dtype=torch.float16):
                hs = model(input_ids=ids, attention_mask=att, output_hidden_states=True).hidden_states[layer]
            gm = ((ids > eos_id) & (att == 1)).unsqueeze(-1)       # gene tokens only (special ids are 0..3)
            out[idx] = ((hs.float() * gm).sum(1) / gm.sum(1).clamp(min=1)).cpu().numpy()
    return out

In [ ]:
import os, sys, json, time, pickle, warnings
sys.path.insert(0, "/kaggle/working")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np, pandas as pd, scipy.sparse as sp
import scanpy as sc, anndata as ad
import torch
from IPython.display import display

import scbench as sb
from scbench import CFG, SEED, SEEDS

sb.set_seeds()
OUT = CFG["out_dir"]                         # results (CSV, figures) -> saved with the notebook output
CACHE, DATA = "/kaggle/temp/cache", "/kaggle/temp/data"   # large files stay out of the saved output
for d in (OUT, CACHE, DATA):
    os.makedirs(d, exist_ok=True)
sc.settings.figdir = OUT
sc.set_figure_params(dpi=90, frameon=False)

def cached(csv, fn):
    """Run fn() and save its DataFrame, or reload it if the CSV already exists."""
    if os.path.exists(csv):
        print("loading cached", os.path.basename(csv))
        return pd.read_csv(csv)
    df = fn()
    df.to_csv(csv, index=False)
    return df

# --- environment report ---
import platform, importlib.metadata as md_
print("python", sys.version.split()[0], "|", platform.platform())
for p in ["scanpy", "anndata", "numpy", "pandas", "scikit-learn", "harmonypy", "leidenalg", "igraph",
          "scikit-misc", "scvi-tools", "transformers", "huggingface-hub"]:
    print(f"{p:16s} {md_.version(p)}")
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## 1. PBMC3k sanity check (single batch)

PBMC3k has one batch, so Harmony has nothing to correct. As a **null control** Harmony is run on random pseudo-batches: a sound pipeline should leave the biology metrics essentially unchanged.

**Caveat:** the PBMC3k labels come from the Scanpy tutorial's clustering of this same data with a classical pipeline, which favours classical methods. This is a sanity check, not a fair comparison.

In [ ]:
adata_pbmc = sb.load_pbmc3k()
print(adata_pbmc)
print(adata_pbmc.obs["cell_type"].value_counts())

pbmc = sb.preprocess(adata_pbmc)
rng = np.random.default_rng(SEED)
pbmc.obs["pseudo_batch"] = rng.choice(["A", "B", "C"], size=pbmc.n_obs)
sb.run_harmony(pbmc, batch_key="pseudo_batch", basis="X_pca", out_key="X_pca_harmony")

rows = []
for name, rep in {"pca": "X_pca", "pca_harmony_null": "X_pca_harmony"}.items():
    sb.graph_umap(pbmc, rep, name)
    rows.append(sb.evaluate_bio(pbmc, rep, name, resolutions=CFG["pbmc_resolutions"]))
results_pbmc = pd.DataFrame(rows).set_index("method").round(3)
display(results_pbmc)
results_pbmc.to_csv(f"{OUT}/pbmc3k_baseline_metrics.csv")
sb.plot_umaps(pbmc, ["pca", "pca_harmony_null"], colors=("cell_type",), save="pbmc3k_umap.png")

## 2. Pancreas data: download, audit, exclusions

Source: scIB integration datasets (Luecken et al., *Nature Methods* 2022), figshare article 12420968, file `human_pancreas_norm_complexBatch.h5ad`. The download is verified against figshare's MD5.

**The `counts` layer is not uniform raw counts.** A per-batch audit found three kinds of data:

| Batch | Cells | Values in `counts` layer | Decision |
|---|---|---|---|
| inDrop1-4 | 8,569 | integer UMI counts (median library size about 5k) | used |
| celseq, celseq2 | 3,289 | non-integer, smallest value 1.001958, consistent with a UMI-collision correction | used, rounded to integers (max change 0.495) |
| smartseq2 | 2,394 | integer read counts (median library size about 288k) | used, flagged as read counts, not UMIs |
| fluidigmc1, smarter | 2,130 | TPM/RPKM-like values | **excluded** |

scVI needs count-like input and every method should see identical cells, hence the exclusion. No cell type is lost.

In [ ]:
ARTICLE_ID, FNAME = 12420968, "human_pancreas_norm_complexBatch.h5ad"
PAN_PATH = f"{DATA}/{FNAME}"
sb.figshare_download(ARTICLE_ID, FNAME, PAN_PATH)

adata_pan = sb.load_pancreas(PAN_PATH, batch_col="tech", label_col="celltype",
                             drop_batches=["fluidigmc1", "smarter"], round_batches=["celseq", "celseq2"])
print(adata_pan)

lib = np.asarray(adata_pan.layers["counts"].sum(1)).ravel()
display(pd.DataFrame({"batch": adata_pan.obs["batch"].values, "libsize": lib})
        .groupby("batch").agg(n_cells=("libsize", "size"), median_libsize=("libsize", "median")))
ct = pd.crosstab(adata_pan.obs["cell_type"], adata_pan.obs["batch"])
display(ct.assign(total=ct.sum(axis=1)).sort_values("total", ascending=False))
print("cell types in only ONE batch:", ct.index[(ct > 0).sum(axis=1) == 1].tolist())
print("cell types with < 30 cells  :", ct.index[ct.sum(axis=1) < 30].tolist())

## 3. Classical arm: PCA and PCA + Harmony

Shared preprocessing for **every** method: batch-aware `seurat_v3` HVGs (2,000) on raw counts, `normalize_total` (1e4), `log1p`, scaling, 50-component PCA. Harmony uses default parameters on the 50 PCs. Metrics follow scIB: Leiden resolution chosen by best NMI, ARI/NMI, cell-type ASW, batch ASW, graph connectivity, plus kNN and class-balanced logistic-regression macro-F1 (stratified CV) and leave-one-batch-out macro-F1. Replicates: 3 seeds (Leiden and CV folds).

In [ ]:
pan = sb.preprocess(adata_pan, hvg_batch_key="batch", max_pct_mt=None)   # scIB data is already QC'd
json.dump(list(pan.var_names), open(f"{OUT}/pancreas_hvgs.json", "w"))
sb.run_harmony(pan, batch_key="batch")

classical = {"pca": "X_pca", "pca_harmony": "X_pca_harmony"}
for name, rep in classical.items():
    sb.graph_umap(pan, rep, name)
ps_classical = cached(f"{OUT}/pancreas_classical_per_seed.csv",
                      lambda: pd.concat([sb.evaluate_all(pan, rep, name) for name, rep in classical.items()],
                                        ignore_index=True))
display(sb.summarize(ps_classical)[2].T)

## 4. scVI arm

Standard scvi-tools integration settings, not tuned (2 layers, 30 latent dimensions, negative binomial likelihood, `batch` as batch key), trained on the raw counts of the same 2,000 HVGs. Three training seeds, each evaluated with the matching seed, so the sd includes training variance. Early stopping bounds the runtime (max 400 epochs).

*Known mismatch:* scVI assumes UMI-like counts, but `smartseq2` holds read counts and `celseq`/`celseq2` were rounded.

In [ ]:
import scvi
SCVI_PARAMS = dict(n_layers=2, n_latent=30, gene_likelihood="nb")
MAX_EPOCHS, PATIENCE = 400, 20

cnt = pan.layers["counts"]
d_ = cnt.data if sp.issparse(cnt) else np.asarray(cnt).ravel()
assert d_.min() >= 0 and np.allclose(d_[:200000], np.round(d_[:200000])), "counts are not integer"
a_sv = ad.AnnData(X=cnt.copy(), obs=pan.obs[["batch"]].copy(), var=pd.DataFrame(index=pan.var_names))
scvi.model.SCVI.setup_anndata(a_sv, batch_key="batch")

lat = lambda s: f"{OUT}/pancreas_scvi_latent_s{s}.npy"
scvi_log = []
for s in SEEDS:
    if os.path.exists(lat(s)):
        pan.obsm[f"X_scvi_s{s}"] = np.load(lat(s)); continue
    sb.set_seeds(s); scvi.settings.seed = s
    model = scvi.model.SCVI(a_sv, **SCVI_PARAMS)
    t0 = time.time()
    model.train(max_epochs=MAX_EPOCHS, early_stopping=True, early_stopping_patience=PATIENCE)
    Z = model.get_latent_representation()
    scvi_log.append({"seed": s, "epochs": len(model.history["elbo_train"]), "minutes": round((time.time() - t0) / 60, 1)})
    pan.obsm[f"X_scvi_s{s}"] = Z
    np.save(lat(s), Z)
    del model; torch.cuda.empty_cache()
if scvi_log:
    display(pd.DataFrame(scvi_log))

def eval_scvi():
    fr = []
    for s in SEEDS:
        sb.ensure_graph(pan, f"X_scvi_s{s}", f"scvi_s{s}", umap=(s == SEEDS[0]))
        d = sb.evaluate_all(pan, f"X_scvi_s{s}", f"scvi_s{s}", seeds=(s,))   # eval seed = training seed
        d["method"] = "scvi"
        fr.append(d)
    return pd.concat(fr, ignore_index=True)

ps_scvi = cached(f"{OUT}/pancreas_scvi_per_seed.csv", eval_scvi)
sb.ensure_graph(pan, "X_scvi_s0", "scvi_s0", umap=True)       # needed for later figures even when cached
display(ps_scvi[["seed", "best_res", "n_clusters", "ARI", "NMI", "kNN_macroF1", "LR_macroF1", "LR_LOBO_macroF1"]])

## 5. Foundation-model arm: zero-shot Geneformer V2-104M

**Why an own implementation:** the official `geneformer` package does not import under transformers 5.x (`SpecialTokensMixin` was removed), so only the dictionary files and the model weights are downloaded and the input encoding is re-implemented. The tokenisation follows the official tokenizer's logic (counts normalised by cell total, scaled to 10,000, divided by the gene median, genes ranked in descending order, `<cls>` ... `<eos>`, 4,096 positions). The pooling choices (mean over gene tokens of the second-to-last layer) follow the extractor's documented options but were **not verified against the official code**. This is a re-implementation, not the official package.

**Gene input:** main run = all genes that map to the model vocabulary (how the model is meant to be used, a deliberate departure from the shared-HVG rule); sensitivity run = the same 2,000 HVGs as every other method. Inference is deterministic, so replicates only vary Leiden and the CV folds.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import BertModel

REPO = "ctheodoris/Geneformer"
GF_CODE, GF_MODEL = "/kaggle/temp/Geneformer_code", "/kaggle/temp/Geneformer_model"
dict_names = ["gene_name_id_dict", "token_dictionary", "gene_median_dictionary", "ensembl_mapping_dict"]
snapshot_download(REPO, local_dir=GF_CODE, allow_patterns=[f"geneformer/{n}_gc104M.pkl" for n in dict_names])
snapshot_download(REPO, local_dir=GF_MODEL, allow_patterns=["Geneformer-V2-104M/*"])
MODEL_DIR = f"{GF_MODEL}/Geneformer-V2-104M"
name2id, tokdict, mediandict, ens_map = [pickle.load(open(f"{GF_CODE}/geneformer/{n}_gc104M.pkl", "rb")) for n in dict_names]
gf_cfg = json.load(open(f"{MODEL_DIR}/config.json"))
MAX_LEN = gf_cfg["max_position_embeddings"]
CLS_ID, EOS_ID, PAD_ID = tokdict["<cls>"], tokdict["<eos>"], tokdict["<pad>"]
print({k: gf_cfg[k] for k in ("hidden_size", "num_hidden_layers", "num_attention_heads", "max_position_embeddings", "vocab_size")})

# ---- gene symbols -> Ensembl IDs -> model vocabulary (coverage report) ----
adata_fm = adata_pan[pan.obs_names].copy()           # same cells, same order as every other arm
assert adata_fm.obs_names.equals(pan.obs_names)
gm = sb.build_gene_mapping(adata_fm.var_names, name2id, ens_map, tokdict, mediandict)
genes, ens, in_vocab = gm["genes"], gm["ens"], gm["in_vocab"]
C = sp.csr_matrix(adata_fm.layers["counts"])
n_counts = np.asarray(C.sum(1)).ravel().astype(np.float32)       # total counts per cell, all genes
tot = np.asarray(C.sum(0)).ravel()
print(f"in vocabulary: direct {gm['ens_direct'].isin(set(tokdict)).sum()} -> with alias pass {in_vocab.sum()} of {len(genes)} genes"
      f" | counts covered: {tot[in_vocab.values].sum() / tot.sum():.3f}")
print("duplicate IDs after alias pass (summed):", ens[in_vocab].duplicated().sum(),
      "| vocabulary genes without a usable median:", int((~gm["valid"]).sum()))

# ---- rank-value encoding, all vocabulary genes ----
tok_all = sb.rank_tokens((C @ gm["M"]).tocsr(), n_counts, gm["tok_ids"], gm["medians"], gm["valid"], CLS_ID, EOS_ID, MAX_LEN)
lens = np.array([len(t) for t in tok_all])
print(f"tokens per cell (incl. cls/eos): min {lens.min()} | median {int(np.median(lens))} | max {lens.max()}"
      f" | truncated at {MAX_LEN}: {(lens == MAX_LEN).mean():.1%}")
print("median tokens per cell by batch:", dict(pd.Series(lens).groupby(pan.obs["batch"].values).median().astype(int)))

# sanity: canonical markers should sit near the top of their cell type's ranking
sym_tok = {g: tokdict[e] for g, e in zip(genes[in_vocab], ens[in_vocab])}
ctv = pan.obs["cell_type"].values
for ct_, g in {"beta": "INS", "alpha": "GCG", "delta": "SST", "gamma": "PPY", "epsilon": "GHRL",
               "acinar": "CELA3A", "ductal": "KRT19", "endothelial": "PLVAP", "macrophage": "CD68"}.items():
    t = sym_tok.get(g)
    if t is None:
        print(f"{ct_:12s} {g:7s} not in vocabulary"); continue
    idx = np.where(ctv == ct_)[0]
    pos = np.array([np.where(tok_all[i] == t)[0][0] for i in idx if (tok_all[i] == t).any()])
    print(f"{ct_:12s} {g:7s} present in {len(pos) / len(idx):4.0%} of cells | median position {np.median(pos) if len(pos) else float('nan'):.0f} (1 = top)")

In [ ]:
gf_model = BertModel.from_pretrained(MODEL_DIR, add_pooling_layer=False).to("cuda").eval()
print("parameters (M):", round(sum(p.numel() for p in gf_model.parameters()) / 1e6, 1))

def gf_embed(name, tokens, bs=8):
    """Embed (or reload from cache) one token set."""
    f = f"{CACHE}/pancreas_{name}.npy"
    if os.path.exists(f):
        return np.load(f)
    t0 = time.time()
    Z = sb.embed_tokens(gf_model, tokens, PAD_ID, EOS_ID, bs=bs)
    np.save(f, Z)
    print(f"{name}: {(time.time() - t0) / 60:.1f} min")
    return Z

pan.obsm["X_gf_all"] = gf_embed("geneformer_all", tok_all)                   # main run: all vocabulary genes

is_hvg = genes.isin(pan.var_names).values                                     # sensitivity run: the shared 2,000 HVGs
M_h = (sp.diags(is_hvg.astype(np.float32)) @ gm["M"]).tocsr()
tok_hvg = sb.rank_tokens((C @ M_h).tocsr(), n_counts, gm["tok_ids"], gm["medians"], gm["valid"], CLS_ID, EOS_ID, MAX_LEN)
lens_h = np.array([len(t) for t in tok_hvg])
print(f"HVG-only tokens per cell: min {lens_h.min()} | median {int(np.median(lens_h))} | max {lens_h.max()}")
assert lens_h.min() > 2, "a cell has no expressed HVG token"
pan.obsm["X_gf_hvg"] = gf_embed("geneformer_hvg", tok_hvg)

reps_gf = {"geneformer_all": "X_gf_all", "geneformer_hvg": "X_gf_hvg"}
for name, rep in reps_gf.items():
    sb.ensure_graph(pan, rep, name, umap=True)
ps_gf = cached(f"{OUT}/pancreas_geneformer_per_seed.csv",
               lambda: pd.concat([sb.evaluate_all(pan, rep, name) for name, rep in reps_gf.items()], ignore_index=True))
display(sb.summarize(ps_gf)[2].T)

### 5b. Is Geneformer's weak result partly an artifact of my choices? Sensitivity checks

1. **Token count.** Does the embedding's main axis track the number of tokens per cell (overall and within each batch)?
2. **Token cap.** Cap every cell at 1,000 tokens (the shortest cell has 749), which removes most length differences.
3. **`TAAR5`.** One gene is rank 1 in many cells because its Genecorpus median is about 100 times below the typical gene and the source file attributes abundant counts to it. The capped run is repeated without it.
4. **Dimension match.** Reduce both main embeddings to 50 PCs (as for PCA and Harmony) and re-score, because a 768-dimensional embedding gives a linear classifier more capacity.

In [ ]:
from sklearn.decomposition import PCA
from scipy.stats import spearmanr

# ---- S0: does the embedding track tokens per cell? (top PCs, then PC1 within each batch) ----
for name, rep, L in [("geneformer_all", "X_gf_all", lens), ("geneformer_hvg", "X_gf_hvg", lens_h)]:
    P = PCA(n_components=5, random_state=SEED).fit_transform(pan.obsm[rep])
    print(name, "| |Spearman| of PC1-PC5 with tokens per cell:", [round(abs(spearmanr(P[:, k], L)[0]), 2) for k in range(5)])
P1 = PCA(n_components=1, random_state=SEED).fit_transform(pan.obsm["X_gf_all"]).ravel()
for b_ in sorted(pan.obs["batch"].unique()):
    m = (pan.obs["batch"] == b_).values
    print(f"{b_:10s} |Spearman| PC1 vs tokens per cell (geneformer_all, within batch): {abs(spearmanr(P1[m], lens[m])[0]):.2f}")

# ---- S1: capped variants (cap 1,000 tokens), with and without TAAR5 ----
e_taar5 = ens[(genes == "TAAR5").values].iloc[0]
valid_noT = gm["valid"].copy(); valid_noT[gm["index"][e_taar5]] = False
Cv_all = (C @ gm["M"]).tocsr()
variants = {"geneformer_cap1000": gm["valid"], "geneformer_cap1000_noTAAR5": valid_noT}
for name, valid in variants.items():
    toks = sb.rank_tokens(Cv_all, n_counts, gm["tok_ids"], gm["medians"], valid, CLS_ID, EOS_ID, 1000)
    pan.obsm[f"X_{name}"] = gf_embed(name, toks, bs=16)
    sb.ensure_graph(pan, f"X_{name}", name, umap=(name == "geneformer_cap1000"))

# ---- S2: 50-PC versions of the main embeddings ----
for src, name in [("X_gf_all", "geneformer_all_pca50"), ("X_gf_hvg", "geneformer_hvg_pca50")]:
    pan.obsm[f"X_{name}"] = PCA(n_components=50, random_state=SEED).fit_transform(pan.obsm[src])
    sb.ensure_graph(pan, f"X_{name}", name, umap=False)

def eval_sens():
    fr = [sb.evaluate_light(pan, f"X_{n}", n) for n in variants]                       # clustering + geometry + kNN
    fr += [sb.evaluate_all(pan, f"X_{n}", n) for n in ("geneformer_all_pca50", "geneformer_hvg_pca50")]
    return pd.concat(fr, ignore_index=True)
ps_sens = cached(f"{OUT}/pancreas_geneformer_sensitivity_per_seed.csv", eval_sens)
print("best Leiden resolution per sensitivity run (grid max is 1.8):")
print(ps_sens.groupby("method")["best_res"].agg(list).to_string())

## 6. Results

In [ ]:
ps_all = pd.concat([ps_classical, ps_scvi, ps_gf], ignore_index=True)
order = ["pca", "pca_harmony", "scvi", "geneformer_all", "geneformer_hvg"]
mean_all, sd_all, table_all = sb.summarize(ps_all)
display(table_all.loc[order].T)
ps_all.to_csv(f"{OUT}/pancreas_all_per_seed_final.csv", index=False)
mean_all.loc[order].to_csv(f"{OUT}/pancreas_summary_mean.csv"); sd_all.loc[order].to_csv(f"{OUT}/pancreas_summary_sd.csv")

# sensitivity table (the capped runs skip the logistic-regression metrics)
sens_order = order + ["geneformer_cap1000", "geneformer_cap1000_noTAAR5", "geneformer_all_pca50", "geneformer_hvg_pca50"]
display(sb.summarize(pd.concat([ps_all, ps_sens], ignore_index=True))[2].loc[sens_order].T)

metrics = ["NMI", "ARI", "kNN_macroF1", "LR_macroF1", "LR_LOBO_macroF1", "cell_type_ASW", "batch_ASW", "graph_conn"]
sb.plot_umaps(pan, ["pca", "pca_harmony", "scvi_s0", "geneformer_all", "geneformer_hvg"], save="pancreas_umap_all_methods.png")
sb.metric_bar(ps_all, metrics, order, save="pancreas_metrics_bar.png")
sb.plot_umaps(pan, ["geneformer_all", "geneformer_cap1000", "geneformer_hvg"], save="pancreas_umap_geneformer_sensitivity.png")

### Saved figure outputs

The following PNGs are preserved from the saved outputs of the supplied original Kaggle notebook. They are stored as standalone assets under `results/figures/` so they render on GitHub. Re-running this notebook regenerates the analysis; the exact UMAP layout or Leiden clustering may vary.

#### Main pancreas embedding comparison

![Pancreas UMAPs across all embedding methods](../results/figures/pancreas_all_methods_umaps.png)

#### Benchmark metrics

![Benchmark metric comparison](../results/figures/pancreas_metrics_comparison.png)

#### Geneformer sensitivity analysis

![Geneformer token-cap sensitivity UMAPs](../results/figures/geneformer_token_cap_sensitivity_umaps.png)

#### Additional saved diagnostics

PBMC3k Harmony null-control outputs:

![PBMC3k null-control output with seven Leiden clusters](../results/figures/pbmc3k_null_control_7_clusters.png)

![PBMC3k null-control output with six Leiden clusters](../results/figures/pbmc3k_null_control_6_clusters.png)

Pancreas PCA versus Harmony:

![Pancreas PCA and Harmony UMAPs](../results/figures/pancreas_pca_harmony_umaps.png)


In [ ]:
# ---- per-cell-type F1 and the ductal diagnostic ----
reps = {"pca": "X_pca", "pca_harmony": "X_pca_harmony", "scvi": "X_scvi_s0", **reps_gf}
pc_all, preds = sb.per_class_table(pan, reps)
display(pc_all.sort_values("n_cells"))
pc_all.to_csv(f"{OUT}/pancreas_per_class_f1.csv")

y_all, b_all = pan.obs["cell_type"].values, pan.obs["batch"].values
acc = {}
for m in reps:
    d = pd.DataFrame({"y": y_all, "b": b_all, "ok": preds[m] == y_all})
    for t in ["acinar", "ductal"]:
        acc[f"{t}_{m}"] = d[d.y == t].groupby("b")["ok"].mean().round(2)
display(pd.DataFrame(acc))
mask = (y_all == "ductal") & np.isin(b_all, ["inDrop2", "inDrop4"])
print("where do the 581 ductal cells of inDrop2/inDrop4 go (kNN prediction)?")
for m in reps:
    print(f"  {m:15s}", pd.Series(preds[m][mask]).value_counts().to_dict())

# ---- per-type graph connectivity and batch mixing (what each method preserved or lost) ----
gname = {"pca": "pca", "pca_harmony": "pca_harmony", "scvi": "scvi_s0",
         "geneformer_all": "geneformer_all", "geneformer_hvg": "geneformer_hvg"}
conn, mix = sb.per_type_connectivity_mixing(pan, reps, gname)
print("per-type graph connectivity (1 = type stays one connected group):"); display(conn)
print("per-type batch mixing (1 = batches well mixed within the type):"); display(mix)
conn.to_csv(f"{OUT}/pancreas_per_type_connectivity.csv"); mix.to_csv(f"{OUT}/pancreas_per_type_batch_mixing.csv")

## 7. Biological interpretation

1. **Marker genes** (Wilcoxon on log-normalised expression): do the labels make biological sense?
2. **The ductal/acinar puzzle.** Harmony and scVI relabel many ductal cells of inDrop2 and inDrop4 as acinar. Are these cells acinar-like by expression (contamination, label noise, doublets) or ductal cells displaced by over-correction?
3. **Rare-type robustness:** annotation scores recomputed without the two smallest types (`t_cell`: 7 cells, `schwann`: 20), because rare types move macro-F1 a lot.

In [ ]:
cts = sorted(pan.obs["cell_type"].unique())
sc.tl.rank_genes_groups(pan, "cell_type", method="wilcoxon", use_raw=True)
res = {ct: sc.get.rank_genes_groups_df(pan, group=ct).reset_index(drop=True) for ct in cts}
top = pd.DataFrame({ct: res[ct]["names"].head(8).values for ct in cts}).T
display(top); top.to_csv(f"{OUT}/pancreas_top_markers.csv")
canon = {"alpha": "GCG", "beta": "INS", "delta": "SST", "gamma": "PPY", "epsilon": "GHRL", "acinar": "PRSS1",
         "ductal": "KRT19", "activated_stellate": "COL1A1", "quiescent_stellate": "RGS5", "endothelial": "PLVAP",
         "macrophage": "CD68", "mast": "TPSAB1", "schwann": "SOX10", "t_cell": "CD3E"}
print("rank of the canonical marker among all genes for its own cell type (1 = best):")
for ct, g in canon.items():
    pos = res[ct].index[res[ct]["names"] == g]
    print(f"{ct:20s} {g:7s}", int(pos[0]) + 1 if len(pos) else "not in gene set")

In [ ]:
# ---- the ductal/acinar puzzle: are the relabelled cells acinar-like by expression? ----
raw = pan.raw
def module_score(genes_):
    present = [g for g in genes_ if g in raw.var_names]
    return np.asarray(raw.X[:, [raw.var_names.get_loc(g) for g in present]].mean(1)).ravel(), present

acinar_score, used_a = module_score(["PRSS1", "CPA1", "CELA3A", "CTRB1", "CELA2A", "PNLIP", "CPB1"])
ductal_score, used_d = module_score(["KRT19", "SOX9", "CFTR", "TACSTD2", "KRT7"])
print("acinar module:", used_a, "| ductal module:", used_d)
duct, few = y_all == "ductal", np.isin(b_all, ["inDrop2", "inDrop4"])
print("median acinar-module score of ductal cells, by batch:")
print(pd.Series(acinar_score[duct]).groupby(b_all[duct]).median().round(3).to_string())
for m in ["pca_harmony", "scvi"]:
    mis, kept = duct & few & (preds[m] == "acinar"), duct & few & (preds[m] == "ductal")
    print(f"\n{m}: inDrop2/4 ductal cells relabelled acinar (n={mis.sum()}) vs kept ductal (n={kept.sum()}) vs ductal in other batches")
    for label, score in [("acinar module", acinar_score), ("ductal module", ductal_score), ("total counts", n_counts)]:
        print(f"   {label:14s} median {np.median(score[mis]):9.3f} | {np.median(score[kept]):9.3f} | {np.median(score[duct & ~few]):9.3f}")

# ---- rare-type robustness: macro-F1 over the 12 types with >= 30 cells ----
big = pc_all.index[pc_all["n_cells"] >= 30]
print("\ndropped types:", [c for c in pc_all.index if c not in big])
print("macro-F1 excluding them (seed 0, stratified CV):")
print(pc_all.loc[big].drop(columns="n_cells").mean().round(3).to_string())
lobo_rob = {m: round(sb.annot_lobo(pan.obsm[rep], y_all, b_all, labels=list(big)), 3) for m, rep in reps.items()}
print("leave-one-batch-out macro-F1 excluding them:", lobo_rob)
pd.Series(lobo_rob).to_csv(f"{OUT}/pancreas_lobo_excluding_rare_types.csv")

## 8. Conclusions

*These statements describe the run reported in the README (one dataset, one foundation model, zero-shot only). Regenerate them if you change settings.*

**Answer to the title question:** no. Zero-shot Geneformer V2-104M did not beat the classical pipeline on this benchmark; it was last on every clustering and geometry metric. scVI was the best method on the unsupervised metrics.

| Question | Winner | Notes |
|---|---|---|
| Cluster recovery (NMI, ARI) | **scVI** (NMI 0.914, ARI 0.943) | Harmony 0.862 / 0.876, PCA 0.741 / 0.561, Geneformer 0.494 / 0.375 (all genes) and 0.582 / 0.433 (HVG) |
| Cell types stay connected (graph connectivity) | **scVI** (0.989) | Geneformer 0.711 (all genes) |
| Batch mixing (batch ASW, lenient metric) | Harmony 0.876 ≈ scVI 0.870 | PCA 0.831 |
| kNN annotation | **scVI** (0.910) | the advantage over PCA and Harmony comes almost entirely from `epsilon` (31 cells) |
| Supervised annotation (balanced LR, mixed batches) | **plain PCA** (0.949) | Geneformer 0.932 / 0.940 only at 768 dimensions; at 50 PCs 0.836 / 0.843 |
| Cross-batch transfer (leave-one-batch-out) | **depends on the two rarest types** | 14 types: Harmony 0.877 > PCA 0.869 > scVI 0.857 ≈ Geneformer-HVG 0.856 > Geneformer 0.798. 12 types: scVI 0.910 > Harmony 0.899 > PCA 0.884 > Geneformer-HVG 0.869 > Geneformer 0.798 |

**Negative and unexpected results**

- Integration (Harmony, scVI) buys cleaner clustering and batch mixing, but its benefit for supervised and cross-batch annotation is small (at most about 0.03) and not robust to the two rarest cell types. A classifier trained across batches can absorb batch effects itself.
- Geneformer's embedding is dominated by the number of genes expressed per cell: PC1 correlates 0.96-0.98 (Spearman) with token count within five of seven batches. Capping every cell at 1,000 tokens did **not** rescue it, and the `TAAR5` artifact made no material difference.
- Geneformer's embeddings do contain linearly decodable cell-type information (balanced LR 0.93-0.94 at 768 dimensions), but it is not organised in the dominant geometry used by clustering and kNN.
- Harmony and scVI relabel 159 and 108 of the 581 ductal cells in inDrop2/inDrop4 as acinar. Those cells express acinar genes strongly, so this may reflect label noise, ambient RNA or doublets as much as integration failure. This is unresolved.

**Limitations:** one dataset; one foundation model, zero-shot only, via an own re-implementation of the encoding; Harmony and scVI run with default parameters; batch ASW and graph connectivity are own implementations of the scIB definitions, not yet cross-checked against `scib-metrics`; Leiden grid edges (Harmony's best resolution is the grid minimum, the capped Geneformer runs hit the grid maximum); rare types make macro-F1 noisy; the `smartseq2` batch holds read counts, not UMIs.

In [ ]:
print("files to download from the notebook output and commit to the repository:")
for f in sorted(os.listdir(OUT)):
    print(f"  results/{f}  ({os.path.getsize(os.path.join(OUT, f)) / 1e6:.1f} MB)")